# Benchmark — Buril contra los mejores limpiadores de trazos (Google Colab)

Cuaderno **driver fino**: no copia la lógica, sólo llama al script del repo
`tools/benchmark_limpiar_trazos.py`. Así, cuando cambie el modelo o la métrica,
el cuaderno ya los usa en el próximo `git pull`.

**Qué hace**: baja **Buril** de la release `modelo-buril` y el dataset `.npz` de
`dataset-buril` (ambos **anónimos**, sin token), toma N dibujos del
`split=validacion` y les pasa **los mismos** a cada modelo. Después normaliza la
salida de todos (**umbral + esqueleto Zhang-Suen**, para no comparar un esqueleto
de 1 px contra una línea gruesa) y mide IoU/Dice esqueleto↔esqueleto, IoU de
máscara, precisión/recall, **clDice** (topología) y **PSNR**, más latencia y peso.

**Rivales (5)** con los que se compara a Buril:

| Modelo | Tarea | Licencia |
|---|---|---|
| Simo-Serra et al. (`bobbens/sketch_simplification`, GAN) | limpieza de bocetos (referencia) | **CC-BY-NC-SA (no comercial)** |
| Anime2Sketch | extracción de línea | MIT (código) |
| Informative Drawings | generación de líneas (no apareada) | MIT (código) |
| ControlNet Lineart | lineart desde boceto | Apache-2.0 |
| PidiNet soft-edge | bordes suaves | Apache-2.0 |

Más dos referencias: **Zhang-Suen** puro (algoritmo) y, si la release tiene el
`.tflite`, **Buril**.

> **Aviso honesto**: Buril es un *especialista en esqueleto de 1 px* (parches
> 64×64, ~cientos de KB, corre en el móvil); los rivales son limpieza/extracción
> general en *full-image* con pesos de cientos de MB. El protocolo (mismo set +
> misma normalización) hace la comparación justa en *forma*, pero **no hay un
> ganador universal**: mirá todas las métricas, no sólo una. El modelo de
> Simo-Serra es **no comercial**; sirve como benchmark de investigación, no como
> argumento de venta.

**Antes de empezar**: `Entorno de ejecución → Cambiar tipo de entorno → GPU`
(opcional pero recomendado). No hace falta token.

In [ ]:
# @title 1. Configuración y clonado del repo
REPO = "LuisAlejandro544/Dibujo-"           # @param {type:"string"}
RAMA = "main"                               # @param {type:"string"}
TAG_MODELO = "modelo-buril"                 # @param {type:"string"}
TAG_DATOS = "dataset-buril"                 # @param {type:"string"}
VERSION_DATOS = "v1.0.0-experimental.1a"    # @param {type:"string"}
LADO = 512                                  # @param {type:"integer"}
POR_CATEGORIA = 100                         # @param {type:"integer"}
MUESTRAS = 8                                # @param {type:"integer"}
SEMILLA = 1                                 # @param {type:"integer"}
MODELOS = "buril,simoserra,anime2sketch,informative,lineart,pidinet,zhangsuen"  # @param {type:"string"}

import os, sys, subprocess

RUTA = "/content/Dibujo-"
if os.path.isdir(os.path.join(RUTA, ".git")):
    subprocess.run(["git", "fetch", "origin", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "checkout", RAMA], cwd=RUTA, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", RAMA], cwd=RUTA, check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", RAMA,
         f"https://github.com/{REPO}.git", RUTA], check=True)

os.chdir(RUTA)
print("Repositorio listo en", RUTA, "| rama", RAMA)

In [ ]:
# @title 2. Dependencias
import importlib, subprocess, sys

def _instalar(*paquetes):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *paquetes], check=True)

for modulo, paquete in [
    ("numpy", "numpy"), ("PIL", "pillow"), ("skimage", "scikit-image"),
    ("matplotlib", "matplotlib"), ("tensorflow", "tensorflow"),
    ("torch", "torch"), ("torchvision", "torchvision"),
    ("controlnet_aux", "controlnet-aux"), ("huggingface_hub", "huggingface_hub"),
]:
    try:
        importlib.import_module(modulo)
    except ImportError:
        print("instalando", paquete, "…")
        _instalar(paquete)

import tensorflow as tf
import torch
print("TensorFlow", tf.__version__, "| torch", torch.__version__,
      "| GPU:", torch.cuda.is_available())

In [ ]:
# @title 3. Bajar Buril y el dataset desde las releases (anónimo, sin token)
import os
import requests

API = f"https://api.github.com/repos/{REPO}"


def assets(tag):
    r = requests.get(f"{API}/releases/tags/{tag}")
    return r.json().get("assets", []) if r.status_code == 200 else []


def bajar(tag, nombre, destino):
    url = f"https://github.com/{REPO}/releases/download/{tag}/{nombre}"
    r = requests.get(url, stream=True)
    if r.status_code != 200:
        return False
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    with open(destino, "wb") as f:
        for chunk in r.iter_content(1 << 20):
            f.write(chunk)
    print(f"Bajado {nombre} ({os.path.getsize(destino)} bytes) de {tag}")
    return True


BURIL = ".cache/modelos/limpiar_trazos.tflite"
if not bajar(TAG_MODELO, "limpiar_trazos.tflite", BURIL):
    print("AVISO: la release no tiene el modelo; Buril se saltea.")
    BURIL = None

DIR = ".cache/limpiar_trazos_planb/procesado"
NPZ = f"{DIR}/pares_esq_lado{LADO}.npz"
ASSET = f"pares_esq_lado{LADO}_pc{POR_CATEGORIA}_{VERSION_DATOS}.npz"
disponibles = [a["name"] for a in assets(TAG_DATOS)]
objetivo = ASSET if ASSET in disponibles else next(
    (n for n in disponibles
     if n.startswith(f"pares_esq_lado{LADO}_pc") and n.endswith(".npz")), None)
if objetivo is None or not bajar(TAG_DATOS, objetivo, NPZ):
    raise SystemExit(
        f"No hay .npz en la release {TAG_DATOS}. "
        "Corré tools/buril_colab.ipynb para generarlo.")
print("Dataset:", NPZ, "|", os.path.getsize(NPZ), "bytes")

In [ ]:
# @title 4. Correr el benchmark y ver las imágenes en línea
import os, sys, importlib

sys.path.insert(0, os.path.join(RUTA, "tools"))
import benchmark_limpiar_trazos as bm
importlib.reload(bm)

from IPython.display import Image, display

ids = [s.strip() for s in MODELOS.split(",") if s.strip()]
corrida = bm.ejecutar(NPZ, BURIL, n=MUESTRAS, semilla=SEMILLA, ids=ids)

bm.imprimir_ranking(corrida["informe"])

if corrida["predicciones"]:
    salida = os.path.join(RUTA, ".cache/benchmark")
    bm.guardar(corrida["informe"], None, salida)
    try:
        ruta_rejilla = bm.dibujar_rejilla(
            corrida["entradas"], corrida["gt"], corrida["predicciones"],
            os.path.join(salida, "rejilla.png"),
            max_muestras=min(4, MUESTRAS))
        display(Image(ruta_rejilla))
    except Exception as e:
        print("No se pudo dibujar la rejilla:", type(e).__name__, e)
else:
    print("No se pudo evaluar ningún modelo.")

In [ ]:
# @title 5. (Opcional) Interfaz Gradio para comparar muestra por muestra
import os
import numpy as np

if "corrida" in globals() and corrida["predicciones"]:
    nombres = ["entrada"] + corrida["informe"]["orden"] + ["GT (1 px)"]

    def _ver(a):
        a = np.asarray(a, np.float32)
        if a.max() > 1.0:
            a = a / 255.0
        return (a * 255).astype(np.uint8)

    def _galeria(i):
        i = int(i)
        imgs = ([corrida["entradas"][i]]
                + [corrida["predicciones"][k][i] for k in corrida["informe"]["orden"]]
                + [corrida["gt"][i]])
        return [(_ver(a), t) for a, t in zip(imgs, nombres)]

    try:
        import gradio as gr
    except ImportError:
        import sys, subprocess
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
        import gradio as gr

    with gr.Blocks() as demo:
        gr.Markdown("## Limpieza de trazos — entrada → modelos → ground truth")
        deslizador = gr.Slider(0, max(len(corrida["entradas"]) - 1, 0),
                               step=1, value=0, label="muestra")
        galeria = gr.Gallery(label="comparación")
        demo.load(_galeria, deslizador, galeria)
        deslizador.change(_galeria, deslizador, galeria)
    demo.launch(debug=False, share=True)
else:
    print("Corré primero la celda 4.")

## Cómo leer los resultados

- **clDice** es la métrica más informativa acá: con *ground truth* de 1 px el IoU
  es durísimo y castiga a todos por igual; clDice premia seguir la **conectividad**
  correcta.
- **IoU/Dice esqueleto↔esqueleto** mide la forma fina; **IoU de máscara** favorece
  a las líneas gruesas.
- **Latencia** y **peso** son la ventaja real de Buril (sin red, en el equipo).
- La rejilla y la interfaz Gradio son para inspección visual: una métrica no
  reemplaza el ojo.

## Próximos pasos

Con las métricas reales se puede cerrar el pendiente de `ROADMAP.md`
("ajustar épocas/semilla") y decidir si conviene ajustar el `--parches-por-imagen`
o la aumentación. Si un rival no carga (por ejemplo, por pesos movidos), el
cuaderno lo reporta y sigue con los demás.